# AtlasEstate 3D - Spatial Analysis & Geodesic Gradients
## Spatial Point Pattern Analysis, POI Proximity & District Stratification
**Author:** AtlasEstate Engineering Team  
**Date:** March 2026  
**Projection:** EPSG:4326 (WGS 84)

---
### Objectives
1. Inspect spatial distribution of 1,290 verified Marrakech listings.
2. Compute distance-decay metrics against major urban amenities (Medina, Airport, Gare, Carre Eden).
3. Evaluate spatial autocorrelation and local price variance across 13 administrative districts.
4. Prepare spatial features for downstream machine learning valuation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

DATA_DIR = Path("../data")
df = pd.read_parquet(DATA_DIR / "gold" / "gold_properties_enriched.parquet")
with open(DATA_DIR / "geospatial" / "marrakech_neighborhoods.geojson", "r", encoding="utf-8") as f:
    neighborhoods_geojson = json.load(f)

with open(DATA_DIR / "geospatial" / "marrakech_pois.geojson", "r", encoding="utf-8") as f:
    pois_geojson = json.load(f)

print(f"Loaded {len(df)} properties, {len(neighborhoods_geojson['features'])} district polygons, and {len(pois_geojson['features'])} POIs.")


### 1. Spatial Scatterplot of Listings Colored by Price/m²


In [ ]:
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    df['longitude'], 
    df['latitude'], 
    c=df['price_per_m2'], 
    cmap='viridis', 
    alpha=0.65, 
    s=24,
    edgecolor='none'
)
cbar = plt.colorbar(scatter)
cbar.set_label("Price per m² (MAD)")
plt.title("Spatial Distribution of Marrakech Properties Colored by Price/m²", fontsize=13, fontweight='bold')
plt.xlabel("Longitude (°E)")
plt.ylabel("Latitude (°N)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### 2. Distance Decay: Proximity to Historic City Center vs Price/m²
Properties situated closer to the historic center (Medina / Jemaa el-Fnaa) and the European commercial core (Gueliz) command strong location premiums.


In [ ]:
plt.figure(figsize=(10, 5))
sns.regplot(
    x='dist_to_center_km', 
    y='price_per_m2', 
    data=df, 
    scatter_kws={'alpha': 0.35, 'color': '#6366f1', 's': 18},
    line_kws={'color': '#ef4444', 'linewidth': 2}
)
plt.title("Distance Decay: Price per m² vs Distance to City Center (km)", fontsize=13, fontweight='bold')
plt.xlabel("Distance to City Center (km)")
plt.ylabel("Price / m² (MAD)")
plt.show()


### 3. District Density & Geographic Median Pricing


In [ ]:
district_metrics = df.groupby('neighborhood').agg(
    listing_count=('property_id', 'count'),
    median_price=('price', 'median'),
    median_price_per_m2=('price_per_m2', 'median'),
    mean_dist_center=('dist_to_center_km', 'mean')
).sort_values(by='median_price_per_m2', ascending=False)

district_metrics.style.format({
    'listing_count': '{:,.0f}',
    'median_price': '{:,.0f} MAD',
    'median_price_per_m2': '{:,.0f} MAD/m²',
    'mean_dist_center': '{:.2f} km'
})


### 4. Key Spatial Insights
1. **Centrifugal Price Decay:** A strong inverse correlation exists between distance to Gueliz/Hivernage and property price/m².
2. **Palmeraie Luxury Outlier:** Palmeraie breaks the purely linear distance decay, representing an ultra-luxury sprawling oasis district with high absolute listing values.
3. **Spatial Feature Utility:** Inclusion of geodesic distance vectors (`dist_to_center_km`, `dist_to_transit_km`, `dist_to_shopping_km`) provides high predictive signal for valuation.
